In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("runid", "2026")
runid = dbutils.widgets.get("runid")

# Data TER

In [0]:
# Lecture des données
df_ter = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"/Volumes/transport/bronze/files/regularite_mensuelle_ter-{runid}.csv")
)

In [0]:
# Formatage des colonnes
df_ter = (
    df_ter
    .withColumnRenamed("Date", "date")
    .withColumnRenamed("Région", "region")
    .withColumnRenamed("Nombre de trains programmés", "nb_trains_programmes")
    .withColumnRenamed("Nombre de trains ayant circulé", "nb_trains_circules")
    .withColumnRenamed("Nombre de trains annulés", "nb_trains_annules")
    .withColumnRenamed("Nombre de trains en retard à l'arrivée", "nb_trains_retards")
    .withColumnRenamed("Taux de régularité", "taux_regul")
    .withColumnRenamed("Nombre de trains à l'heure pour un train en retard à l'arrivée", "ratio_trains_a_l_heure")
    .withColumnRenamed("Commentaires", "commentaires")
    .withColumn("runid", F.lit(runid))
)

In [0]:
# Stockage
(
    df_ter.write
    .format('delta')
    .mode('overwrite')
    .partitionBy('runid')
    .option("overwriteSchema", "true")
    .saveAsTable("transport.silver.regularite_ter")
)

# Data TGV

In [0]:
# Lecture des données
df_tgv = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"/Volumes/transport/bronze/files/regularite_mensuelle_tgv-{runid}.csv")
)

In [0]:
# Formatage des colonnes
df_tgv = (
    df_tgv
    .withColumnRenamed("Date", "date")
    .withColumnRenamed("Service", "service")
    .withColumnRenamed("Gare de départ", "gare_depart")
    .withColumnRenamed("Gare d'arrivée", "gare_arrivee")
    .withColumnRenamed("Durée moyenne du trajet", "duree_moyenne_trajet")
    .withColumnRenamed("Nombre de circulations prévues", "nb_circulations_prevues")
    .withColumnRenamed("Nombre de trains annulés", "nb_trains_annules")
    .withColumnRenamed("Commentaire annulations", "commentaire_annulations")
    .withColumnRenamed("Nombre de trains en retard au départ", "nb_trains_retards_depart")
    .withColumnRenamed("Retard moyen des trains en retard au départ", "retard_moyen_trains_retards_depart")
    .withColumnRenamed("Retard moyen de tous les trains au départ", "retard_moyen_tous_trains_depart")
    .withColumnRenamed("Commentaire retards au départ", "commentaire_retards_depart")
    .withColumnRenamed("Nombre de trains en retard à l'arrivée", "nb_trains_retards_arrivee")
    .withColumnRenamed("Retard moyen des trains en retard à l'arrivée", "retard_moyen_trains_retards_arrivee")
    .withColumnRenamed("Retard moyen de tous les trains à l'arrivée", "retard_moyen_tous_trains_arrivee")
    .withColumnRenamed("Commentaire retards à l'arrivée", "commentaire_retards_arrivee")
    .withColumnRenamed("Nombre trains en retard > 15min", "nb_trains_retards_15min")
    .withColumnRenamed("Retard moyen trains en retard > 15 (si liaison concurrencée par vol)", "retard_moyen_trains_retards_15min")
    .withColumnRenamed("Nombre trains en retard > 30min", "nb_trains_retards_30min")
    .withColumnRenamed("Nombre trains en retard > 60min", "nb_trains_retards_60min")
    .withColumnRenamed("Prct retard pour causes externes", "prct_retard_causes_externes")
    .withColumnRenamed("Prct retard pour cause infrastructure", "prct_retard_cause_infrastructure")
    .withColumnRenamed("Prct retard pour cause gestion trafic", "prct_retard_cause_gestion_trafic")
    .withColumnRenamed("Prct retard pour cause matériel roulant", "prct_retard_cause_mat_roulant")
    .withColumnRenamed("Prct retard pour cause gestion en gare et réutilisation de matériel", "prct_retard_cause_gestion_gare_mat")
    .withColumnRenamed("Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)", "prct_retard_cause_prise_compte_voyageurs")
    .withColumn("runid", F.lit(runid))
)

In [0]:
# Stockage
(
    df_tgv.write
    .format('delta')
    .mode('overwrite')
    .partitionBy('runid')
    .option("overwriteSchema", "true")
    .saveAsTable("transport.silver.regularite_tgv")
)

# Data transilien

In [0]:
# Lecture des données
df_transilien = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"/Volumes/transport/bronze/files/regularite_mensuelle_transilien-{runid}.csv")
)

In [0]:
# Formatage des colonnes
df_transilien = (
    df_transilien
    .withColumnRenamed("Date", "date")
    .withColumnRenamed("Service", "service")
    .withColumnRenamed("Ligne", "ligne")
    .withColumnRenamed("Nom de la ligne", "nom_ligne")
    .withColumnRenamed("Taux de ponctualité", "taux_regul")
    .withColumnRenamed("Nombre de voyageurs à l'heure pour un voyageur en retard", "ratio_trains_a_l_heure")
    .withColumn("runid", F.lit(runid))
)

In [0]:
# Stockage
(
    df_transilien.write
    .format('delta')
    .mode('overwrite')
    .partitionBy('runid')
    .option("overwriteSchema", "true")
    .saveAsTable("transport.silver.regularite_transilien")
)

# Data gare voyageur

In [0]:
# Lecture des données
df_gare = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"/Volumes/transport/bronze/files/gares_de_voyageurs-{runid}.csv")
)

In [0]:
df_gare.display()

In [0]:
# Formatage des colonnes
df_gare = (
    df_gare
    .withColumnRenamed("Nom_Gare", "nom_gare")
    .withColumnRenamed("Trigramme", "trigramme")
    .withColumnRenamed("Segment(s) DRG", "segment_drg")
    .withColumnRenamed("Position géographique", "position_géographique")
    .withColumnRenamed("Code commune", "code_commune")
    .withColumnRenamed("Code_UIC", "code_uic")
    .withColumnRenamed("Id_Gare", "id_gare")
    .withColumn("runid", F.lit(runid))
)

In [0]:
(
    df_gare.write
    .format('delta')
    .mode('overwrite')
    .partitionBy('runid')
    .option("overwriteSchema", "true")
    .saveAsTable("transport.silver.gares")
)